This notebook shows a couple of examples of using this package to fit large behavioral datasets. The examples don't exactly correspond to the actual analyses in the papers (in part because the statistical package has come a long way!) but it gives the basic idea.

This is the Python/JAX ([pyem](https://github.com/ndawlab/em-jax)) version of the EM.jl example notebook, so the likelihood functions are rewritten in JAX.

In [ ]:
###### setup

# this allows threading over CPU cores (change number as appropriate); must come before importing jax
import os
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=16"

import jax
jax.config.update("jax_enable_x64", True)  # required -- JAX defaults to float32
import jax.numpy as jnp
from jax.scipy.special import erf, logsumexp
import numpy as np
import pandas as pd
from scipy import stats
import urllib.request

import pyem as em

full = False    # Maintain full covariance matrix (vs a diagonal one) at the group level
emtol = 1e-3    # stopping condition (relative change) for EM

<h1>Patent race example</h1>

Hunter, L. E., Meer, E. A., Gillan, C. M., Hsu, M., & Daw, N. D. (2022). Increased and biased deliberation in social anxiety. Nature Human Behaviour, 6(1), 146-154.

This paper used repeated play of a patent race game to study the relationship between counterfactual updating and social anxiety. Here we fit the combined data from both experiments in the paper.

In [ ]:
# EWA model likelihood functions from Hunter et al. (2020), rewritten in JAX
# (these are variants on Camerer and Ho's EWA learning model)
# the original julia is at https://github.com/ndawlab/patentrace/blob/main/julia/patent%20likfuns.jl
# simplified from the paper by omitting the free initial value params (which are overkill): Q starts at 7

# payoff matrix: GAME[i, j] is the payoff for move i+1 when the opponent plays j+1
GAME = jnp.array([[5 - i + (10 if i > j else 0) for j in range(1, 7)] for i in range(1, 6)], dtype=float)

# the likelihood function is specified by a step function that is like the inner part of the loop over
# trials: it takes the current state and the player's and opponent's moves, and returns the new state
# and the log-likelihood of the choice
def ewa_step_maker(beta, phi, wplus, wminus):
    rho = phi

    def step(state, s1, s2):  # s1: player's move (1-5), s2: opponent's move (1-6)
        Q, N = state
        ll = beta * Q[s1 - 1] - logsumexp(beta * Q)  # choice likelihood: log of softmax
        Nnew = rho * N + 1
        # learning step for every move j: the chosen move learns from its payoff,
        # unchosen moves from their foregone payoff weighted by w+ (if better than
        # what was obtained) or w- (if not)
        payoff = GAME[:, s2 - 1]
        w = jnp.where(jnp.arange(5) == s1 - 1, 1.0, jnp.where(payoff > payoff[s1 - 1], wplus, wminus))
        return ((phi * N * Q + w * payoff) / Nnew, Nnew), ll

    return (jnp.full(5, 7.0), jnp.array(0.0)), step

squash = lambda x: 0.5 + 0.5 * erf(x / jnp.sqrt(2.0))  # unit normal CDF, to 0-1

# you make a likelihood function by applying the scan_likelihood decorator to the step function
# (here we have two variants that differ in the number of parameters)

# full (valenced EWA) model: params [beta, phi, w+, w-]
@em.scan_likelihood
def likfun_ewa_valenced(params):
    return ewa_step_maker(params[0], squash(params[1]), squash(params[2]), squash(params[3]))

# baseline model is same as above, but with w+ = w-: params [beta, phi, w]
@em.scan_likelihood
def likfun_ewa(params):
    return ewa_step_maker(params[0], squash(params[1]), squash(params[2]), squash(params[2]))

# data
patent_data = pd.read_csv("https://raw.githubusercontent.com/ndawlab/patentrace/refs/heads/main/julia/patent_investments_743subs.csv")
patent_covars = pd.read_csv("https://raw.githubusercontent.com/ndawlab/patentrace/refs/heads/main/julia/mergedcovars.csv")

# subject order shared by the design matrix and the per-trial data
patentsubs = list(patent_covars["sub"])
assert set(patentsubs) == set(patent_data["sub"])
patentNS = len(patentsubs)

In [ ]:
# fit 3 parameter model (basic EWA) with IQ and LSAS as covariates

# set up the design matrix: intercept and 2 covariates (IQ and LSAS)
X = jnp.array(np.column_stack([np.ones(patentNS), patent_covars.iqZ, patent_covars.lsasZ]))  # design matrix

# starting params (one row per covariate, one column per parameter)
startbetas = jnp.array([[1., 0, 0], [0, 0, 0], [0, 0, 0]])
startsigma = jnp.array([5., 1, 1])

# set up the model and fit it
model_ewa = em.EMModel(patent_data, X, cols=("s1", "s2"), subs=patentsubs, likfun=likfun_ewa, nparam=3,
                       reg_names=["Intercept", "IQz", "LSASz"], param_names=["beta", "phi", "w"])
fit_ewa = em.em_fit(model_ewa, startbetas, startsigma, emtol=emtol, full=full)
fit_ewa

In [ ]:
# test significance
# IQ correlates with everything, LSAS specifically picks up w (counterfactual/MB learning)

em.emerrors(fit_ewa)

In [ ]:
# fit alternative model: separates upward and downward counterfactual learning (w+ and w-)

startbetas = jnp.array([[1., 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]])
startsigma = jnp.array([5., 1, 1, 1])

model_valenced = em.EMModel(patent_data, X, cols=("s1", "s2"), subs=patentsubs, likfun=likfun_ewa_valenced, nparam=4,
                            reg_names=["Intercept", "IQz", "LSASz"], param_names=["beta", "phi", "w+", "w-"])
fit_valenced = em.em_fit(model_valenced, startbetas, startsigma, emtol=emtol, full=full)
fit_valenced

In [ ]:
# here the LSAS effect is specific to w+ (upward counterfactual learning)
em.emerrors(fit_valenced)

In [ ]:
# model comparison: we can compare the two models in aggregate level 
# using iAIC which is quick and indicates the more complicated model
# fits better
print("iAIC for baseline model: ", float(em.iaic(fit_ewa)))
print("iAIC for valenced model: ", float(em.iaic(fit_valenced)))

In [ ]:
# but it is better (though slow!) to compute the likelihood per subject
# so as to test whether the difference is significant taking account of
# subject-subject variation in fit. To do that we need to compute the
# cross-validated likelihoods (LOOCV) per subject and model, whch is really 
# slow and then we can test the paired difference in fits

print("\nFitting first model")
liks_ewa = em.loocv(fit_ewa, emtol=emtol)
print("\nFitting second model")
liks_valenced = em.loocv(fit_valenced, emtol=emtol)

# test the difference in cross-validated likelihoods across subjects (paired t test)
stats.ttest_rel(liks_ewa, liks_valenced)

<h1>Two-step task example</h1>
Gillan, C. M., Kosinski, M., Whelan, R., Phelps, E. A., & Daw, N. D. (2016). Characterizing a psychiatric symptom dimension related to deficits in goal-directed control. elife, 5, e11305.

This paper looked at how psychiatric symptoms (and other individual differences) predict learning in a two-step Markov decision task. The main finding is that a transdiagnostic compulsivity factor ("Factor 2") selectively predicts model-based learning, relative to two other factors (Factors 1 & 3, depression and social anxiety)

In [ ]:
# load the data
# work around slightly irritating princeton anti-bot measures

url = "https://www.princeton.edu/~ndaw/gillan_trials.csv"
url2 = "https://www.princeton.edu/~ndaw/gillan_subjects.csv"

req = urllib.request.Request(
    url, 
    headers={'User-Agent': 'Mozilla/5.0'}
)

gillandata = pd.read_csv(urllib.request.urlopen(req))

req2 = urllib.request.Request(
    url2,
    headers={'User-Agent': 'Mozilla/5.0'}
)
gillancovars = pd.read_csv(urllib.request.urlopen(req2))

subs = list(gillandata["sub"].unique())

NS = len(subs)

In [ ]:
# the EM package has a sample likelihood function for this task (hybrid MB/MF), em.seqlik_nll, which is
# between the model from Gillan et al. (2017) and the one from Daw et al (2011),
# in that it omits decay
#
# this is a model variant which includes decay of the unchosen options' values, more like
# the full Gillan model

def softmaximum(a, b):
    # differentiable approximation to the max over two actions
    p = 1.0 / (1.0 + jnp.exp(-5 * (a - b)))
    return p * a + (1 - p) * b

# again you make the likelihood function by applying the scan_likelihood decorator to a step function
# which is like the inner part of a loop over trials: it takes the current state and information about
# the current trial, and returns the trial likelihood and the new state
#
# note confusingly "state" here is the state of the model (Q values, etc.) which gets carried over the
# loop, and not the per trial second-stage state (here called s) the subject encounters
@em.scan_likelihood
def seqlik_decay(params):
    # this is more or less the likelihood function for the two-step decision task from Gillan et al., eLife 2015

    beta1m, beta1t0, beta1t1, beta2 = params[0], params[1], params[2], params[3]
    lr = 0.5 + 0.5 * erf(params[4] / jnp.sqrt(2.0))
    ps = params[5]

    def step(state, c1, c2, mn, s):
        Q0, Q1, prevc = state
        i1, si = c1 - 1, s - 1
        r = 2 * mn - 1  # code rewards as -1,1 to center decay

        Qm = jnp.array([softmaximum(Q0[1, 0], Q0[1, 1]), softmaximum(Q0[2, 0], Q0[2, 1])])
        Qd = beta1m * Qm + beta1t0 * Q0[0, :] + beta1t1 * Q1
        Qd = jnp.where(prevc > 0, Qd.at[jnp.maximum(prevc - 1, 0)].add(ps), Qd)  # perseveration
        ll1 = Qd[i1] - logsumexp(Qd)

        # the second-stage choice can be missing even when the first is not:
        # then count only the first-stage choice and skip learning
        valid2 = c2 > 0
        i2 = jnp.where(valid2, c2 - 1, 0)
        ll2 = jnp.where(valid2, beta2 * Q0[si, i2] - logsumexp(beta2 * Q0[si, :]), 0.0)

        Q0d, Q1d = (1 - lr) * Q0, (1 - lr) * Q1  # decay all Q values
        Q0d = Q0d.at[0, i1].add(Q0d[si, i2])
        Q1d = Q1d.at[i1].add(r)
        Q0d = Q0d.at[si, i2].add(r)

        return (jnp.where(valid2, Q0d, Q0), jnp.where(valid2, Q1d, Q1), c1.astype(jnp.int32)), ll1 + ll2

    return (jnp.zeros((3, 2)), jnp.zeros(2), jnp.zeros((), dtype=jnp.int32)), step

In [ ]:
# fit basic hybrid MB / MF model with covariates

# set up the design matrix: intercept and 6 covariates (3 factors, age, IQ, gender)
X = jnp.array(np.column_stack([np.ones(NS), gillancovars.Factor1, gillancovars.Factor2, gillancovars.Factor3,
                               gillancovars.agez, gillancovars.iqz, gillancovars.gender]))  # design matrix

# starting params (one row per covariate, one column per parameter)
startbetas = jnp.zeros((7, 6)).at[0, :].set(jnp.array([1., 1., 1., 1., 0, 1]))
startsigma = jnp.array([5., 5, 5, 5, 1, 5])

# set up the model and fit it
reg_names = ["Intercept", "Factor1", "Factor2", "Factor3", "Agez", "Iqz", "Gender"]
param_names = ["betaMB", "betaTD0", "betaTD1", "beta2", "lr", "ps"]

model = em.EMModel(gillandata, X, cols=("ch1", "ch2", "mn", "st"), subs=subs, likfun=em.seqlik_nll, nparam=6,
                   reg_names=reg_names, param_names=param_names)
fit = em.em_fit(model, startbetas, startsigma, emtol=emtol, full=full)
fit

In [ ]:
# individual differences effects
# the paper mainly focused on betaMB (model-based control),
# which decreases with compulsion (Factor2) & age, & increases with IQ
# but there are lots of other things here, eg other params changing with age

em.emerrors(fit)

In [ ]:
# fit model with decay

model_decay = em.EMModel(gillandata, X, cols=("ch1", "ch2", "mn", "st"), subs=subs, likfun=seqlik_decay, nparam=6,
                         reg_names=reg_names, param_names=param_names)
fit_decay = em.em_fit(model_decay, startbetas, startsigma, emtol=emtol, full=full)
fit_decay

In [ ]:
# individual differences effects
# (pretty similar to the other model)

em.emerrors(fit_decay)

In [ ]:
# again a quick group-level model comparison suggests the decay model is better
print("iAIC for baseline model: ", float(em.iaic(fit)))
print("iAIC for decay model: ", float(em.iaic(fit_decay)))

In [ ]:
# but it is better (though slower) to do per-subject cross-validated comparison
# computing the likelihood per subject so as to test whether the difference 
# is significant taking account of subject-subject variation in fit.
# this will maybe take an hour for a dataset this size
print("\nFitting first model")
liks_baseline = em.loocv(fit, emtol=emtol)
print("\nFitting second model")
liks_decay = em.loocv(fit_decay, emtol=emtol)

# test the difference in cross-validated likelihoods across subjects (paired t test)
stats.ttest_rel(liks_baseline, liks_decay)